# 5.17 — Understanding Data Leakage Through Simple Examples

**Data leakage** happens when training (or evaluation) accidentally uses information that would not exist at prediction time.

In this notebook you will see 3 common leakage patterns and their fixes:

1. **Target leakage** (a feature is derived from the target)
2. **Train–test contamination** (fitting preprocessing on all data)
3. **Aggregated leakage** (computing group statistics using full data)

The repo already applies the main protection rules:

- Split before fitting anything (see `split_data(...)`)
- Fit preprocessing only on `X_train` (see `fit_preprocessor(...)`)
- Keep a clear list of excluded/leakage columns (see `EXCLUDED_COLUMNS`)

In [ ]:
import numpy as np
import pandas as pd

from src.preprocessing import split_data, fit_preprocessor, transform_features
from src.model import train_model
from src.evaluate import evaluate_model
from src.config import DEFAULT_RANDOM_STATE

## Helper: fit on train only (the correct pattern)

This helper follows the **no-leakage** workflow:

1. Split data into train/test
2. Fit preprocessor on `X_train` only
3. Transform `X_train` and `X_test` using the fitted preprocessor
4. Train on transformed `X_train`
5. Evaluate on transformed `X_test`

In [ ]:
def fit_train_evaluate(
    X: pd.DataFrame,
    y: pd.Series,
    *,
    test_size: float = 0.2,
    random_state: int = DEFAULT_RANDOM_STATE,
    time_column: str | None = None,
) -> dict[str, float | str]:
    X_train, X_test, y_train, y_test = split_data(
        X,
        y,
        test_size=test_size,
        random_state=random_state,
        time_column=time_column,
    )

    bundle = fit_preprocessor(X_train)
    X_train_t = transform_features(X_train, bundle)
    X_test_t = transform_features(X_test, bundle)

    model = train_model(X_train_t, y_train)
    return evaluate_model(model, X_test_t, y_test)

## Example 1 — Target leakage (feature derived from target)

We simulate a churn dataset and add a “too good to be true” feature:

- `cancellation_reason` is **only present when churn already happened**.

This is leakage because at the moment you need to predict churn, you do not know the cancellation reason.

In [ ]:
rng = np.random.default_rng(42)

n = 1200

tenure = rng.integers(1, 60, size=n)
monthly_charges = rng.normal(50, 15, size=n).clip(10, 150)
contract_type = rng.choice(["month_to_month", "one_year", "two_year"], size=n, p=[0.6, 0.25, 0.15])

# True churn mechanism (some signal, but not perfect)
logit = (
    -0.02 * tenure
    + 0.015 * monthly_charges
    + (contract_type == "month_to_month") * 0.8
)
p = 1 / (1 + np.exp(-logit))
y = (rng.random(n) < p).astype(int)
y = pd.Series(y, name="churn")

X_base = pd.DataFrame(
    {
        "tenure": tenure,
        "monthly_charges": monthly_charges,
        "contract_type": contract_type,
    }
)

# LEAKY FEATURE: directly derived from the target
cancellation_reason = np.where(y.to_numpy() == 1, "customer_cancelled", "not_cancelled")
X_leaky = X_base.copy()
X_leaky["cancellation_reason"] = cancellation_reason

metrics_leaky = fit_train_evaluate(X_leaky, y)
metrics_clean = fit_train_evaluate(X_base, y)

print("With target leakage feature:")
print(f"  accuracy={metrics_leaky['accuracy']:.4f}  f1={metrics_leaky['f1_score']:.4f}")
print("Without target leakage feature:")
print(f"  accuracy={metrics_clean['accuracy']:.4f}  f1={metrics_clean['f1_score']:.4f}")

## Example 2 — Train–test contamination (fitting preprocessing on all data)

A subtle leakage pattern:

- If you **fit** a scaler/encoder/imputer using the full dataset before splitting, then information from the test distribution influences training transformations.

Below we intentionally do it wrong, then compare against the correct approach.

In [ ]:
# A dataset with a distribution shift (simulating time drift)
# First 80% has lower mean charges; last 20% has higher mean charges.
rng = np.random.default_rng(7)

n = 1000
cut = int(n * 0.8)

charges = np.concatenate([
    rng.normal(40, 10, size=cut),
    rng.normal(70, 10, size=n - cut),
]).clip(10, 150)

region = rng.choice(["north", "south", "east", "west"], size=n)

# Target depends on charges, but noise makes it non-trivial
p = 1 / (1 + np.exp(-(0.06 * (charges - 55))))
y = (rng.random(n) < p).astype(int)

df = pd.DataFrame({"monthly_charges": charges, "region": region})
y = pd.Series(y, name="target")

# CORRECT: split first, fit on train only
metrics_correct = fit_train_evaluate(df, y)

# WRONG: fit preprocessing on the full dataset, then split
bundle_wrong = fit_preprocessor(df)  # leakage: sees future/test distribution
X_all_t = transform_features(df, bundle_wrong)
X_train_t, X_test_t, y_train, y_test = split_data(X_all_t, y, test_size=0.2, random_state=42)
model_wrong = train_model(X_train_t, y_train)
metrics_wrong = evaluate_model(model_wrong, X_test_t, y_test)

print("Correct (fit preprocessor on train only):")
print(f"  accuracy={metrics_correct['accuracy']:.4f}  f1={metrics_correct['f1_score']:.4f}")
print("Wrong (fit preprocessor on full data before split):")
print(f"  accuracy={metrics_wrong['accuracy']:.4f}  f1={metrics_wrong['f1_score']:.4f}")

## Example 3 — Aggregated leakage (group statistics computed using full data)

A common feature idea is a group statistic like:

- `regional_target_rate` = mean of the target for that region

This is only valid if the statistic is computed **using training data only**.

Below:

- **Leaky version** computes the rate using all rows (including test labels).
- **Correct version** computes the rate on training rows, then maps it onto test rows.

In [ ]:
rng = np.random.default_rng(123)

n = 2500
region = rng.choice(["A", "B", "C", "D", "E"], size=n, p=[0.1, 0.2, 0.3, 0.25, 0.15])
base_risk = pd.Series(region).map({"A": 0.15, "B": 0.25, "C": 0.35, "D": 0.45, "E": 0.55}).to_numpy()

noise = rng.normal(0, 1, size=n)
other_feature = noise

p = (base_risk + 0.05 * (other_feature > 0)).clip(0.02, 0.98)
y = (rng.random(n) < p).astype(int)
y = pd.Series(y, name="target")

X = pd.DataFrame({"region": region, "other_feature": other_feature})

# -----------------
# Leaky aggregation
# -----------------
X_leaky = X.copy()
X_leaky["regional_target_rate"] = pd.DataFrame({"region": region, "y": y}).groupby("region")["y"].transform("mean")

metrics_leaky_agg = fit_train_evaluate(X_leaky, y)

# -----------------
# Correct aggregation
# -----------------
X_train, X_test, y_train, y_test = split_data(X, y, test_size=0.2, random_state=42)

train_rates = (
    pd.DataFrame({"region": X_train["region"], "y": y_train})
    .groupby("region")["y"]
    .mean()
)
global_fallback = float(y_train.mean())

X_train_fixed = X_train.copy()
X_test_fixed = X_test.copy()

X_train_fixed["regional_target_rate"] = X_train_fixed["region"].map(train_rates).fillna(global_fallback)
X_test_fixed["regional_target_rate"] = X_test_fixed["region"].map(train_rates).fillna(global_fallback)

bundle = fit_preprocessor(X_train_fixed)
X_train_t = transform_features(X_train_fixed, bundle)
X_test_t = transform_features(X_test_fixed, bundle)

model = train_model(X_train_t, y_train)
metrics_fixed_agg = evaluate_model(model, X_test_t, y_test)

print("Leaky aggregation (uses full dataset labels):")
print(f"  accuracy={metrics_leaky_agg['accuracy']:.4f}  f1={metrics_leaky_agg['f1_score']:.4f}")
print("Correct aggregation (train-only stats mapped to test):")
print(f"  accuracy={metrics_fixed_agg['accuracy']:.4f}  f1={metrics_fixed_agg['f1_score']:.4f}")

## Takeaways (the rules to memorize)

- **Prediction-moment test:** at the moment a real prediction is made, would this feature exist?
- **Split first:** do not let test data influence anything that is “fit”.
- **Train-only statistics:** scalers, imputers, feature selectors, and aggregations must be computed using training data only.

In this repo, the core boundary is enforced by:

- `split_data(...)` (train/test separation)
- `fit_preprocessor(X_train)` + `transform_features(X_test, fitted_bundle)` (no contamination)
- `EXCLUDED_COLUMNS` (explicitly remove leakage-prone columns)